In [3]:
!pip install lightgbm

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)


In [4]:
# C:/IDEAL_Programming/src/train_lightgbm.py
# ---------------------------------------------------------
# LightGBM training (IDEAL) — compatible with current pipeline
# Trains:
# 1) lgbm_coldstart
# 2) lgbm_withhistory
#
# Saves:
# - lgbm_coldstart.joblib
# - lgbm_coldstart_feature_columns.json
# - lgbm_coldstart_train_medians.json
# - lgbm_coldstart_train_extras.json
# - lgbm_withhistory.joblib
# - lgbm_withhistory_feature_columns.json
# - lgbm_withhistory_train_medians.json
# - lgbm_withhistory_train_extras.json
# - training_config_lgbm.json
# ---------------------------------------------------------

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, Any

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor


# =========================
# CONFIG
# =========================
BASE_DIR = Path("C:/IDEAL_Programming")
DATA_PATH = BASE_DIR / "processed" / "final" / "IDEAL_final_hourly_dataset.csv"
OUT_DIR = BASE_DIR / "processed" / "models" / "final_lgbm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

TEST_DAYS = 30
RANDOM_STATE = 42
SPLIT_MODE = "global"

USE_LOG_TARGET = True
CLIP_TARGET_TRAIN = True
CLIP_Q_LOW = 0.005
CLIP_Q_HIGH = 0.995

CATEGORICAL = ["hometype", "urban_rural_class"]

NUMERIC_CORE = [
    "hour",
    "day_of_week",
    "is_weekend",
    "month",
    "external_temperature",
    "total_floor_area_m2",
    "residents",
]

LAG_FEATURES = ["lag_1h", "lag_24h", "roll_mean_24h", "roll_mean_168h"]

# σχετικά συντηρητικές τιμές για να μην overfit εύκολα
LGBM_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    objective="regression",
    random_state=RANDOM_STATE,
    n_jobs=4,
    verbosity=-1,
)


# =========================
# METRICS
# =========================
def safe_mape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    den = np.where(np.abs(y_true) < eps, np.nan, y_true)
    return float(np.nanmean(np.abs((y_true - y_pred) / den)) * 100)


def compute_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = safe_mape(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape}


# =========================
# HELPERS
# =========================
def add_time_features(df):
    df["hour"] = df[TIME_COL].dt.hour
    df["day_of_week"] = df[TIME_COL].dt.dayofweek
    df["month"] = df[TIME_COL].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    return df


def clean_types(df):
    num_cols = [
        "external_temperature", "total_floor_area_m2", "residents",
        "hour", "day_of_week", "month", "is_weekend"
    ]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[TARGET_COL]).copy()

    for c in CATEGORICAL:
        df[c] = df[c].astype("string").fillna("Unknown")

    return df


def make_onehot(df, feature_cols, categorical_cols):
    return pd.get_dummies(df[feature_cols], columns=categorical_cols, drop_first=False)


def align_test_to_train(X_train, X_test):
    return X_test.reindex(columns=X_train.columns, fill_value=0)


def time_split_global(df, test_days=30):
    cutoff = df[TIME_COL].max() - pd.Timedelta(days=test_days)
    train_df = df[df[TIME_COL] <= cutoff].copy()
    test_df = df[df[TIME_COL] > cutoff].copy()
    return train_df, test_df, cutoff


def time_split_per_home(df, test_days=30):
    df = df.sort_values([HOME_COL, TIME_COL]).copy()
    cutoffs = df.groupby(HOME_COL)[TIME_COL].max() - pd.Timedelta(days=test_days)
    df = df.join(cutoffs.rename("cutoff"), on=HOME_COL)
    train_df = df[df[TIME_COL] <= df["cutoff"]].copy()
    test_df = df[df[TIME_COL] > df["cutoff"]].copy()
    train_df.drop(columns=["cutoff"], inplace=True)
    test_df.drop(columns=["cutoff"], inplace=True)
    return train_df, test_df, None


def split_data(df):
    if SPLIT_MODE == "per_home":
        return time_split_per_home(df, TEST_DAYS)
    return time_split_global(df, TEST_DAYS)


def train_only_impute(train_df, test_df, cols):
    medians = {}
    for c in cols:
        med = float(train_df[c].median()) if train_df[c].notna().any() else 0.0
        medians[c] = med
        train_df[c] = train_df[c].fillna(med)
        test_df[c] = test_df[c].fillna(med)
    return train_df, test_df, medians


def maybe_clip_train_target(y_train):
    if not CLIP_TARGET_TRAIN:
        return y_train, None
    lo = np.quantile(y_train, CLIP_Q_LOW)
    hi = np.quantile(y_train, CLIP_Q_HIGH)
    y_train_clipped = np.clip(y_train, lo, hi)
    return y_train_clipped, (float(lo), float(hi))


def y_transform_train(y):
    if USE_LOG_TARGET:
        return np.log1p(np.maximum(y, 0.0))
    return y


def y_inverse_pred(y_hat):
    if USE_LOG_TARGET:
        return np.expm1(y_hat)
    return y_hat


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


# =========================
# LOAD DATA
# =========================
raw_cols = [
    HOME_COL, TIME_COL, TARGET_COL,
    "external_temperature", "total_floor_area_m2", "residents",
    "hometype", "urban_rural_class"
]

df = pd.read_csv(DATA_PATH, usecols=raw_cols, parse_dates=[TIME_COL], low_memory=False)
df = df.dropna(subset=[TARGET_COL]).copy()
df = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)

df = add_time_features(df)
df = clean_types(df)

df_global_time = df.sort_values(TIME_COL).reset_index(drop=True)


# =========================
# MODE 1: COLD-START
# =========================
FEATURES_COLD = NUMERIC_CORE + CATEGORICAL
train_cold, test_cold, cutoff = split_data(df_global_time)

num_cols_cold = [c for c in NUMERIC_CORE if c not in CATEGORICAL]
train_cold, test_cold, medians_cold = train_only_impute(train_cold, test_cold, num_cols_cold)

for c in num_cols_cold:
    train_cold[c] = train_cold[c].astype("float32")
    test_cold[c] = test_cold[c].astype("float32")

train_cold[TARGET_COL] = train_cold[TARGET_COL].astype("float32")
test_cold[TARGET_COL] = test_cold[TARGET_COL].astype("float32")

X_train_cold = make_onehot(train_cold, FEATURES_COLD, CATEGORICAL).astype("float32")
X_test_cold = make_onehot(test_cold, FEATURES_COLD, CATEGORICAL).astype("float32")
X_test_cold = align_test_to_train(X_train_cold, X_test_cold).astype("float32")

y_train_cold = train_cold[TARGET_COL].to_numpy(dtype="float32")
y_test_cold = test_cold[TARGET_COL].to_numpy(dtype="float32")

y_train_cold_used, clip_bounds_cold = maybe_clip_train_target(y_train_cold)
y_train_cold_tr = y_transform_train(y_train_cold_used)

lgbm_cold = LGBMRegressor(**LGBM_PARAMS)
print(f"\nTraining LGBM (cold-start) | SPLIT_MODE={SPLIT_MODE} | LOG={USE_LOG_TARGET} | CLIP={CLIP_TARGET_TRAIN} ...")
lgbm_cold.fit(X_train_cold, y_train_cold_tr)

pred_cold_tr = lgbm_cold.predict(X_test_cold).astype("float32")
pred_cold = y_inverse_pred(pred_cold_tr).astype("float32")

metrics_cold = compute_metrics(y_test_cold, pred_cold)
print("LGBM cold-start metrics:", metrics_cold)

lgbm_cold_path = OUT_DIR / "lgbm_coldstart.joblib"
joblib.dump(lgbm_cold, lgbm_cold_path)

cold_cols_path = OUT_DIR / "lgbm_coldstart_feature_columns.json"
save_json(list(X_train_cold.columns), cold_cols_path)

cold_medians_path = OUT_DIR / "lgbm_coldstart_train_medians.json"
save_json(medians_cold, cold_medians_path)

cold_extra_path = OUT_DIR / "lgbm_coldstart_train_extras.json"
save_json(
    {
        "split_mode": SPLIT_MODE,
        "test_days": TEST_DAYS,
        "cutoff_global": str(cutoff) if cutoff is not None else None,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds_cold,
        "lgbm_params": LGBM_PARAMS,
        "features_cold": FEATURES_COLD,
    },
    cold_extra_path,
)


# =========================
# MODE 2: WITH-HISTORY
# =========================
df_lags = df.sort_values([HOME_COL, TIME_COL]).reset_index(drop=True)
g = df_lags.groupby(HOME_COL, sort=False)[TARGET_COL]

df_lags["lag_1h"] = g.shift(1)
df_lags["lag_24h"] = g.shift(24)
df_lags["roll_mean_24h"] = g.shift(1).rolling(24).mean()
df_lags["roll_mean_168h"] = g.shift(1).rolling(168).mean()

df_lags = df_lags.dropna(subset=LAG_FEATURES).copy()
for c in LAG_FEATURES:
    df_lags[c] = pd.to_numeric(df_lags[c], errors="coerce")

df_lags_global = df_lags.sort_values(TIME_COL).reset_index(drop=True)

FEATURES_HIST = NUMERIC_CORE + LAG_FEATURES + CATEGORICAL
train_hist, test_hist, cutoff2 = split_data(df_lags_global)

num_cols_hist = [c for c in (NUMERIC_CORE + LAG_FEATURES) if c not in CATEGORICAL]
train_hist, test_hist, medians_hist = train_only_impute(train_hist, test_hist, num_cols_hist)

for c in num_cols_hist:
    train_hist[c] = train_hist[c].astype("float32")
    test_hist[c] = test_hist[c].astype("float32")

train_hist[TARGET_COL] = train_hist[TARGET_COL].astype("float32")
test_hist[TARGET_COL] = test_hist[TARGET_COL].astype("float32")

X_train_hist = make_onehot(train_hist, FEATURES_HIST, CATEGORICAL).astype("float32")
X_test_hist = make_onehot(test_hist, FEATURES_HIST, CATEGORICAL).astype("float32")
X_test_hist = align_test_to_train(X_train_hist, X_test_hist).astype("float32")

y_train_hist = train_hist[TARGET_COL].to_numpy(dtype="float32")
y_test_hist = test_hist[TARGET_COL].to_numpy(dtype="float32")

y_train_hist_used, clip_bounds_hist = maybe_clip_train_target(y_train_hist)
y_train_hist_tr = y_transform_train(y_train_hist_used)

lgbm_hist = LGBMRegressor(**LGBM_PARAMS)
print(f"\nTraining LGBM (with-history) | SPLIT_MODE={SPLIT_MODE} | LOG={USE_LOG_TARGET} | CLIP={CLIP_TARGET_TRAIN} ...")
lgbm_hist.fit(X_train_hist, y_train_hist_tr)

pred_hist_tr = lgbm_hist.predict(X_test_hist).astype("float32")
pred_hist = y_inverse_pred(pred_hist_tr).astype("float32")

metrics_hist = compute_metrics(y_test_hist, pred_hist)
print("LGBM with-history metrics:", metrics_hist)

lgbm_hist_path = OUT_DIR / "lgbm_withhistory.joblib"
joblib.dump(lgbm_hist, lgbm_hist_path)

hist_cols_path = OUT_DIR / "lgbm_withhistory_feature_columns.json"
save_json(list(X_train_hist.columns), hist_cols_path)

hist_medians_path = OUT_DIR / "lgbm_withhistory_train_medians.json"
save_json(medians_hist, hist_medians_path)

hist_extra_path = OUT_DIR / "lgbm_withhistory_train_extras.json"
save_json(
    {
        "split_mode": SPLIT_MODE,
        "test_days": TEST_DAYS,
        "cutoff_global": str(cutoff2) if cutoff2 is not None else None,
        "use_log_target": USE_LOG_TARGET,
        "clip_target_train": CLIP_TARGET_TRAIN,
        "clip_bounds": clip_bounds_hist,
        "lgbm_params": LGBM_PARAMS,
        "features_hist": FEATURES_HIST,
        "lag_features": LAG_FEATURES,
    },
    hist_extra_path,
)

# =========================
# SUMMARY / CONFIG
# =========================
config = {
    "data_path": str(DATA_PATH),
    "target": TARGET_COL,
    "time_col": TIME_COL,
    "home_col": HOME_COL,
    "test_days": TEST_DAYS,
    "split_mode": SPLIT_MODE,
    "use_log_target": USE_LOG_TARGET,
    "clip_target_train": CLIP_TARGET_TRAIN,
    "clip_q_low": CLIP_Q_LOW,
    "clip_q_high": CLIP_Q_HIGH,
    "lgbm_params": LGBM_PARAMS,
    "features_cold_numeric_core": NUMERIC_CORE,
    "features_hist_lags": LAG_FEATURES,
    "categorical": CATEGORICAL,
    "metrics_cold": metrics_cold,
    "metrics_withhistory": metrics_hist,
}
config_path = OUT_DIR / "training_config_lgbm.json"
save_json(config, config_path)

print("\nSaved artifacts:")
print(" -", lgbm_cold_path)
print(" -", cold_cols_path)
print(" -", cold_medians_path)
print(" -", cold_extra_path)
print(" -", lgbm_hist_path)
print(" -", hist_cols_path)
print(" -", hist_medians_path)
print(" -", hist_extra_path)
print(" -", config_path)


Training LGBM (cold-start) | SPLIT_MODE=global | LOG=True | CLIP=True ...
LGBM cold-start metrics: {'MAE': 155.087158203125, 'RMSE': 334.1978385814307, 'MAPE_%': 50.79626206710801}

Training LGBM (with-history) | SPLIT_MODE=global | LOG=True | CLIP=True ...
LGBM with-history metrics: {'MAE': 129.13836669921875, 'RMSE': 304.7134412033706, 'MAPE_%': 36.109216948821846}

Saved artifacts:
 - C:\IDEAL_Programming\processed\models\final_lgbm\lgbm_coldstart.joblib
 - C:\IDEAL_Programming\processed\models\final_lgbm\lgbm_coldstart_feature_columns.json
 - C:\IDEAL_Programming\processed\models\final_lgbm\lgbm_coldstart_train_medians.json
 - C:\IDEAL_Programming\processed\models\final_lgbm\lgbm_coldstart_train_extras.json
 - C:\IDEAL_Programming\processed\models\final_lgbm\lgbm_withhistory.joblib
 - C:\IDEAL_Programming\processed\models\final_lgbm\lgbm_withhistory_feature_columns.json
 - C:\IDEAL_Programming\processed\models\final_lgbm\lgbm_withhistory_train_medians.json
 - C:\IDEAL_Programming\